In [1]:
"""Tutorial: Overlap, Kinetic, and Nuclear Attraction Integrals via Obara-Saika Recursion"""

__authors__   = ["Kien Bui"]
__credits__   = ["Kien Bui"]

__copyright__ = "(c) 2014-2018, The Psi4NumPy Developers"
__license__   = "BSD-3-Clause"
__date__      = "2026-08-10"

# Overlap, Kinetic, and Nuclear Attraction Integrals

Tutorial [11a](11a_1e_Integrals.ipynb) introduced the Obara-Saika (OS) recursion and used it to build the overlap, kinetic energy, and dipole moment integrals. This tutorial revisits overlap and kinetic energy with a different organization of the same recursion, and then extends the scheme to the **nuclear attraction integrals** (NAIs), the last one-electron ingredient of the core Hamiltonian $\mathbf{H} = \mathbf{T} + \mathbf{V}$.

Contents:

1. Notation and the Gaussian product theorem
2. Overlap integrals, and the three-dimensional assembly **product**
3. Kinetic energy integrals, and the three-dimensional assembly **sum of cross terms**
4. The Boys function
5. Nuclear attraction integrals and the auxiliary index $m$
6. Assembling $\mathbf{S}$, $\mathbf{T}$, and $\mathbf{V}$ over a Psi4 basis set, and checking against Psi4

## Notation and the Gaussian product theorem

We work with unnormalized primitive Cartesian Gaussians, as defined in 11a:

\begin{equation}
\phi(\mathbf r; \alpha, \mathbf n, \mathbf A) =
(x - A_x)^{n_x} (y - A_y)^{n_y} (z - A_z)^{n_z} \exp[-\alpha (\mathbf r - \mathbf A)^2]
\end{equation}

For two primitives with exponents $a$ and $b$ on centers $\mathbf A$ and $\mathbf B$, the Gaussian product theorem collapses the product into a single Gaussian on a new, weighted center $\mathbf P$:

\begin{equation}
\zeta = a + b, \qquad \xi = \frac{ab}{\zeta}, \qquad
\mathbf P = \frac{a\mathbf A + b\mathbf B}{\zeta}
\end{equation}



Every recursion below is built on this product Gaussian.

> **A note on symbols.** This tutorial follows the symbol conventions of the original Obara-Saika paper [[Obara:1986](http://aip.scitation.org/doi/abs/10.1063/1.450106)]: $\zeta$ is the **total** exponent $a+b$, and $\xi$ is the **reduced** exponent $ab/\zeta$. Tutorial 11a uses `alpha` for the total exponent and `zeta` for the reduced one. The mathematics is identical, only the names differ.

## Overlap integrals

The two-center overlap integral between two primitive Cartesian Gaussians is

\begin{equation}
S_{ab} = (\mathbf a|\mathbf b) = \int d \mathbf r \phi_a(\mathbf r) \phi_b(\mathbf r)
\tag{[Obara:1986] Eqn A1}
\end{equation}

### The prefactor

The $s$-$s$ overlap, or the base case where both functions have zero angular momentum, is just the Gaussian product prefactor:

\begin{equation}
S_{00} = S_{\text{base}} = (\mathbf 0_A | \mathbf 0_B)
= \left(\frac{\pi}{\zeta}\right)^{3/2} e^{-\xi|\mathbf A - \mathbf B|^2}
\tag{[Obara:1986] Eqn 22}
\end{equation}

$S_{\text{base}}$ depends only on the exponents and the centers, carrying no angular momentum indices. Every higher-angular-momentum overlap is therefore $S_{\text{base}}$ multiplied by a dimensionless factor that depends only on the angular momentum indices. This suggests a clean trick to separate the Cartesian dimension in the arrays:

- the **recursion functions** return only the dimensionless table $\tilde S$, seeded at $\tilde S_{00} = 1$;
- the **prefactor** $S_{\text{base}}$ is computed once per primitive pair, up in the assembly loop and outside of the table. It's multiplied just like the primitive coefficients.

Tutorial 11a makes the same decoupling, using the variable `start` for what we call `S_base` and seeding its recursion at 1.0 for the same reason.

### Separability of the Cartesian axes

The three-dimensional overlap Gaussian factorizes in three Cartesian directions, so it is an exact product of three independent one-dimensional tables:

\begin{equation}
S_{ab} = S_{\text{base}}\;
\tilde S^{(x)}_{n_{ax} n_{bx}}\;
\tilde S^{(y)}_{n_{ay} n_{by}}\;
\tilde S^{(z)}_{n_{az} n_{bz}}
\end{equation}

### The recursion

\begin{equation}
\tilde S^{(x)}_{i,j} = (P_x - A_x)\, \tilde S_{i-1,j}
+ \frac{i-1}{2\zeta}\, \tilde S_{i-2,j}
+ \frac{j}{2\zeta}\, \tilde S_{i-1,j-1}
\tag{[Obara:1986] Eqn A2}
\end{equation}

\begin{equation}
\tilde S^{(x)}_{i,j} = (P_x - B_x)\, \tilde S_{i,j-1}
+ \frac{i}{2\zeta}\, \tilde S_{i-1,j-1}
+ \frac{j-1}{2\zeta}\, \tilde S_{i,j-2}
\tag{[Obara:1986] Eqn A2}
\end{equation}

Here $S^{(x)}_{i,j}$ is the dimensionless overlap along the $x$ axis between $x$-angular-momentum $i$ on center A and $j$ on center B. Any term whose index would go negative is zero.

**Important note**: These are written in **step-down** form ($\tilde S_{i,j}$ expressed via $\tilde S_{i-1,\cdot}$) rather than the step-up form ($\tilde S_{i+1,j}$) of the original paper. The two are algebraically identical, but step-down indexing reads more naturally in the code as we are looping over indices to fill in the current indices ($i$ and $j$). This convention will be used throughout the notebook instead of the conventional step-up notation ($i+1$ and $j$).

### The algorithm

We fill an $(\ell_a{+}1) \times (\ell_b{+}1)$ table in two phases, matching the two relations above:

1. **The $j = 0$ column** is filled first by pure center-A stepping. Each entry depends only on lower-$i$ entries of the same column.
2. **The $j > 0$ entries** are then filled by center-B stepping, each depending on entries already computed in columns $j-1$ and $j-2$.

Additional notes:

- $\mathbf P - \mathbf A$ and $\mathbf P - \mathbf B$ are three-vectors and the recursion is *identical* along each axis, so we can carry all three axes at once by making the leading array dimension the axis index. Every line below operates on a length-3 slice, so NumPy broadcasting evaluates $x$, $y$, and $z$ simultaneously. The table is indexed `SI[axis, i, j]` with `axis` in $\{0, 1, 2\}$.
- `get_overlap_table` returns only the per-axis $\tilde S$ pieces, and the product assembly happens one layer up, in the shell loop.
- Since $j = 0$ throughout the first phase (step up on center A), the relation is simplified:

\begin{equation}
\tilde S^{(x)}_{i,0} = (P_x - A_x)\, \tilde S_{i-1,0}
+ \frac{i-1}{2\zeta}\, \tilde S_{i-2,0}
\end{equation}

In [2]:
import numpy as np
import psi4


def get_overlap_table(PA, PB, zeta, AMa, AMb):
    """Dimensionless Obara-Saika overlap tables for all three Cartesian axes.

    Parameters
    ----------
    PA, PB : ndarray, shape (3,)
        Displacements P - A and P - B of the Gaussian product center.
    zeta : float
        Total exponent, a + b.
    AMa, AMb : int
        Angular momenta of the two shells.

    Returns
    -------
    ndarray, shape (3, AMa + 1, AMb + 1)
        Per-axis table, relative to S_base. 
        NOTE: the caller must assemble these as a product (Sx*Sy*Sz)
    """
    overlap_table = np.zeros((3, AMa + 1, AMb + 1))

    # Seed: the (0|0) case is 1.0 by construction, since S_base is factored out.
    overlap_table[:, 0, 0] = 1.0

    # For a fixed ib we sweep ia, so every entry we read has already been written
    for ib in range(AMb + 1):
        for ia in range(AMa + 1):
            if ia == 0 and ib == 0:
                continue  # already seeded

            if ib == 0:
                # Phase 1: the ib = 0 column, stepping up center A.
                term1 = PA * overlap_table[:, ia - 1, 0]
                term2 = (0.5 / zeta) * (ia - 1) * overlap_table[:, ia - 2, 0] if ia > 1 else 0.0

                overlap_table[:, ia, 0] = term1 + term2

            else:
                # Phase 2: everything else, stepping up center B.
                term1 = PB * overlap_table[:, ia, ib - 1]
                term2 = (0.5 / zeta) * ia * overlap_table[:, ia - 1, ib - 1] if ia > 0 else 0.0
                term3 = (0.5 / zeta) * (ib - 1) * overlap_table[:, ia, ib - 2] if ib > 1 else 0.0

                overlap_table[:, ia, ib] = term1 + term2 + term3

    return overlap_table

## Kinetic energy integrals

The kinetic energy integral is

\begin{equation}
T_{ab} = \left(\mathbf a \left| -\tfrac{1}{2}\nabla^2 \right| \mathbf b\right)
= -\frac{1}{2}\int d\mathbf r\; \phi_a(\mathbf r)\, \nabla^2 \phi_b(\mathbf r)
\tag{[Obara:1986] Eqn A9}
\end{equation}

It shares the same $S_{\text{base}}$ prefactor as the overlap, so we again return a dimensionless per-axis table $\tilde T$. The seed is the one-dimensional $s$-$s$ kinetic shape factor:

### The base case seed

Relative to $S_{\text{base}}$, Helgaker gives the one-dimensional kinetic base case as:

\begin{equation}
\tilde T_{0,0} = a - 2a^2\left(X_{PA}^2 + \frac{1}{2p}\right)
\tag{[Helgaker:2000] Eqn. 9.3.42}
\end{equation}

with $p \equiv \zeta$ and $X_{PA} = P_x - A_x$. Substituting

\begin{equation}
X_{PA} = \frac{aA_x + bB_x}{\zeta} - A_x = -\frac{b}{\zeta}(A_x - B_x)
\end{equation}

gives

\begin{equation}
a - 2a^2\left(\frac{b^2 (A_x-B_x)^2}{\zeta^2} + \frac{1}{2\zeta}\right)
= a\left(1 - \frac a {\zeta}\right) - 2\left(\frac{ab}{\zeta}\right)^2 (A_x-B_x)^2
= \xi\left(1 - 2\xi (A_x - B_x)^2\right)
\end{equation}

\begin{equation}
\tilde T^{(x)}_{0,0} = \xi\left(1 - 2\xi (A_x - B_x)^2\right)
\end{equation}

The second form is preferred as it's symmetric to the two centers. Additionally, as a consistency check, it can easily be seen that summing the seed over the three Cartesian axes recovers the three-dimensional base case quoted in the original Obara-Saika paper:

\begin{equation}
\sum_{k=x,y,z} \xi\left(1 - 2\xi (A_k - B_k)^2\right)
= \xi\left(3 - 2\xi |\mathbf A - \mathbf B|^2\right)
\tag{[Obara:1986] Eqn A13}
\end{equation}

### The recursion

The recursion mirrors the overlap recursion with one extra term:

\begin{equation}
\tilde T_{i,j} = (P_x - A_x)\, \tilde T_{i-1,j}
+ \frac{i-1}{2\zeta}\, \tilde T_{i-2,0}
+ \frac{j}{2\zeta}\, \tilde T_{i-1,j-1}
+ 2\xi\left[\tilde S_{i,j} - \frac{i-1}{2a}\, \tilde S_{i-2,j}\right]
\tag{[Obara:1986] Eqn A12}
\end{equation}

\begin{equation}
\tilde T_{i,j} = (P_x - B_x)\, \tilde T_{i,j-1}
+ \frac{i}{2\zeta}\, \tilde T_{i-1,j-1}
+ \frac{j-1}{2\zeta}\, \tilde T_{i,j-2}
+ 2\xi\left[\tilde S_{i,j} - \frac{j-1}{2b}\, \tilde S_{i,j-2}\right]
\tag{[Obara:1986] Eqn A12}
\end{equation}

The bracketed term is why `get_kinetic_table` takes the overlap table `SI` as an argument, and why the overlap recursion must always be run first.

### The three-dimensional assembly sum of cross terms

The kinetic operator is the Laplacian, $\nabla^2 = \partial_x^2 + \partial_y^2 + \partial_z^2$. It's a **sum** of second derivatives, not a product, so it's not cleanly separable as the overlap. The correct three-dimensional assembly is therefore a sum of three cross terms:

\begin{equation}
T_{ab} = S_{\text{base}}\Big[
\tilde T^{(x)}\tilde S^{(y)}\tilde S^{(z)}
+ \tilde S^{(x)}\tilde T^{(y)}\tilde S^{(z)}
+ \tilde S^{(x)}\tilde S^{(y)}\tilde T^{(z)}
\Big]
\tag{[Helgaker:2000] Eqn 9.3.38}
\end{equation}

(angular momentum indices suppressed for readability).

### The algorithm

The algorithm's structure is essentially identical to computing the overlap integrals. The first phase is stepping up center A where $j = 0$, so the relation is simplified:

\begin{equation}
\tilde T_{i,0} = (P_x - A_x)\, \tilde T_{i-1,0}
+ \frac{i-1}{2\zeta}\, \tilde T_{i-2,0}
+ 2\xi\left[\tilde S_{i,0} - \frac{i-1}{2a}\, \tilde S_{i-2,0}\right]
\end{equation}

It's important to note that `get_kinetic_table` returns only the per-axis $\tilde T$ pieces, and the sum-of-cross-terms assembly happens one layer up, in the shell loop.

In [3]:
def get_kinetic_table(PA, PB, expa, expb, AMa, AMb, SI):
    """Dimensionless Obara-Saika kinetic energy tables for all three Cartesian axes.

    Parameters
    ----------
    PA, PB : ndarray, shape (3,)
        Displacements P - A and P - B.
    expa, expb : float
        Primitive exponents a and b.
    AMa, AMb : int
        Angular momenta of the two shells.
    SI : ndarray, shape (3, AMa + 1, AMb + 1)
        Overlap table from `get_overlap_table`. Required by the final term of the recursion; must be computed first.

    Returns
    -------
    ndarray, shape (3, AMa + 1, AMb + 1)
        Per-axis table, relative to S_base. 
        NOTE: the caller must assemble these as a sum of cross terms (Tx*Sy*Sz + Sx*Ty*Sz + Sx*Sy*Tz)
    """
    zeta = expa + expb
    xi = (expa * expb) / zeta

    kinetic_table = np.zeros((3, AMa + 1, AMb + 1))

    # Seed: the 1D s-s kinetic shape factor, relative to S_base.
    kinetic_table[:, 0, 0] = xi * (1.0 - 2.0 * xi * (PA - PB) ** 2)

    # For a fixed ib we sweep ia, so every entry we read has already been written
    for ib in range(AMb + 1):
        for ia in range(AMa + 1):
            if ia == 0 and ib == 0:
                continue # already seeded

            if ib == 0:
                # Phase 1: the ib = 0 column, stepping up center A.
                term1 = PA * kinetic_table[:, ia - 1, ib]
                term2 = (0.5 / zeta) * (ia - 1) * kinetic_table[:, ia - 2, ib] if ia > 1 else 0.0
                term3 = (0.5 / zeta) * ib * kinetic_table[:, ia - 1, ib - 1] if ib > 0 else 0.0
                term4 = (2 * xi) * (SI[:, ia, ib] - (0.5 / expa) * (ia - 1) * SI[:, ia - 2, ib])

                kinetic_table[:, ia, 0] = term1 + term2 + term3 + term4

            else:
                # Phase 2: everything else, stepping up center B.
                term1 = PB * kinetic_table[:, ia, ib - 1]
                term2 = (0.5 / zeta) * ia * kinetic_table[:, ia - 1, ib - 1] if ia > 0 else 0.0
                term3 = (0.5 / zeta) * (ib - 1) * kinetic_table[:, ia, ib - 2] if ib > 1 else 0.0
                term4 = (2 * xi) * (SI[:, ia, ib] - (0.5 / expb) * (ib - 1) * SI[:, ia, ib - 2])

                kinetic_table[:, ia, ib] = term1 + term2 + term3 + term4

    return kinetic_table

## The Boys function

The Boys function (original definition: [[Boys:1950:542](http://dx.doi.org/10.1098/rspa.1950.0036)]) is involved in every integral involving the Coulomb operator $1/r$ against Gaussian functions:

\begin{equation}
F_m(x) = \int_0^1 t^{2m} e^{-x t^2}\, dt
\tag{[Obara:1986] Eqn 45}
\end{equation}

We use its closed form in terms of the lower incomplete gamma function:

\begin{equation}
F_m(x) = \frac{\Gamma(m+\tfrac12)}{2\, x^{m+1/2}}\, P\!\left(m+\tfrac12,\, x\right),
\qquad P(s,x) = \frac{\gamma(s,x)}{\Gamma(s)}
\tag{[Helgaker:2000] Eqn 9.8.21}
\end{equation}

where $P$ is the *regularized* lower incomplete gamma function, which is exactly what `scipy.special.gammainc` returns. (`gammainc` is already regularized, so no extra division by $\Gamma(s)$ is needed.)

The expression above is $0/0$ indeterminate at $x = 0$, so the limit $F_m(0) = 1/(2m+1)$ is explicitly set for small arguments. This is important to implement as it occurs whenever a nucleus sits exactly on the Gaussian product center, which happens for every one-center integral in the molecule.

In [4]:
from scipy.special import gamma, gammainc


def boys_function(m, x):
    """Boys function F_m(x), via the regularized lower incomplete gamma function."""
    if x < 1e-9:
        # The general expression below is 0/0 at the x = 0 limit.
        return 1.0 / (2 * m + 1)

    return 0.5 * x ** (-(m + 0.5)) * gamma(m + 0.5) * gammainc(m + 0.5, x)

## Nuclear attraction integrals

The nuclear attraction integral between two primitives and a point charge at nucleus $\mathbf C$ is

\begin{equation}
V_{ab} = \left(\mathbf a \left| \frac{1}{|\mathbf r - \mathbf C|} \right| \mathbf b\right)
= \int d\mathbf r\; \frac{\phi_a(\mathbf r)\phi_b(\mathbf r)}{|\mathbf r - \mathbf C|}
\tag{[Obara:1986] Eqn A14}
\end{equation}

### The axes no longer separate

Overlap and kinetic energy factored into per-axis one-dimensional tables because their operators do not couple the Cartesian directions. It is a function of the full
three-dimensional distance, and so it cannot be split into independent $x$, $y$, and $z$ pieces like the overlap or kinetic integrals.

In the OS scheme, an **auxiliary index** $m$ is used, which is the order of the Boys function. Instead of three small 2D tables we now need a 7-dimensional array carrying all six spatial indices plus $m$:

$$(n_{ax},\, n_{ay},\, n_{az},\, n_{bx},\, n_{by},\, n_{bz},\, m)$$

### The prefactor

The Gaussian/Coulomb prefactor is

\begin{equation}
V_{\text{base}} = 2\sqrt{\frac{\zeta}{\pi}}\; S_{\text{base}}
\tag{[Obara:1986] Eqn A20}
\end{equation}

which, like $S_{\text{base}}$, carries no angular momentum indices and is computed once per primitive pair in the assembly loop. The physical matrix element also carries the nuclear charge $-Z_C$, summed over all nuclei.

### The base case seed

The recursion is seeded on the Boys function with argument $U = \zeta|\mathbf P - \mathbf C|^2$:

\begin{equation}
[\mathbf 0|\mathbf 0]^{(m)} = F_m(U), \qquad m = 0, 1, \ldots, \ell_a + \ell_b
\end{equation}

Only $m = 0$ is physically meaningful in the final integral. The higher-$m$ values are pure **scaffolding**: each step up in angular momentum consumes one order of $m$, so building an integral of total angular momentum $L$ requires seeding all orders up to $L$ and then climbing back down to $m = 0$. This is why the seed loop runs to $\ell_a + \ell_b$ and why the code reads out only the `[..., 0]` slice at the end.


### The recursion

Stepping up the $x$ index on center A:

\begin{equation}
[\mathbf a|\mathbf{b}]^{(m)} =
(P_x - A_x)\, [\mathbf a{-}\mathbf 1_x|\mathbf{b}]^{(m)}
- (P_x - C_x)\, [\mathbf a{-}\mathbf 1_x|\mathbf{b}]^{(m+1)}
\end{equation}

\begin{equation}
+ \frac{a_x - 1}{2\zeta}\Big([\mathbf a{-}\mathbf 2_x|\mathbf{b}]^{(m)} - [\mathbf a{-}\mathbf 2_x|\mathbf{b}]^{(m+1)}\Big)
+ \frac{b_x}{2\zeta}\Big([\mathbf a{-}\mathbf 1_x|\mathbf b{-}\mathbf 1_x]^{(m)} - [\mathbf a{-}\mathbf 1_x|\mathbf b{-}\mathbf 1_x]^{(m+1)}\Big)
\tag{[Obara:1986] Eqn A18}
\end{equation}

As noted above, the notation used here is **step-down** rather than **step-up** in the original paper. The other five spatial directions follow the identical pattern with the appropriate displacement vector. Note how every term either keeps $m$ or raises it by one. The recursion only ever reads from **higher** $m$, which explains why the scaffolding is necessary.

### The algorithm

We cannot fill this table axis by axis; the $m$ coupling forces a specific global ordering. Instead we fill it **shell by shell in total angular momentum** $L = 1, 2, \ldots, \ell_a + \ell_b$:

- For each $L$, visit only the index combinations whose six spatial indices sum to exactly $L$;
- For each such combination, fill all orders $m = 0, \ldots, (\ell_a + \ell_b - L)$.

Building in strictly increasing $L$ ensures that every dependency on the right-hand side (which always has total angular momentum $L-1$ or $L-2$) already exists before it is read. The upper limit `max_m - L + 1` on the $m$ loop is important to avoid unnecessary computations. An entry at total angular momentum $L$ will only ever be consumed by entries $L$ steps further up, each of which costs one order of $m$, so orders above $\ell_a + \ell_b - L$ can never be reached and are not worth computing. Within a combination, the code steps up the **first** nonzero spatial index it finds (the six `if`/`elif` branches). Any valid step-down direction yields the same value.

### The index-shifting helper

Written out literally, the recursion above needs a negative-index guard on every one of its terms, and there are six branches of four terms each. Rather than write 24 boundary conditions by hand, an index shifting helper is implemented. Using this makes it much more readable, as each term shifts one or two indices by a one or two and leaves the rest alone. `shift_NAI` captures the current loop indices from the enclosing scope, applies the requested shifts, and returns `0.0` if any index would go negative.

In [5]:
def get_NAI_table(PA, PB, PC, zeta, AMa, AMb):
    """Obara-Saika nuclear attraction integrals for one nucleus, one primitive pair.

    Parameters
    ----------
    PA, PB, PC : ndarray, shape (3,)
        Displacements P - A, P - B, and P - C (C is the nuclear center).
    zeta : float
        Total exponent, a + b.
    AMa, AMb : int
        Angular momenta of the two shells.

    Returns
    -------
    ndarray, shape (AMa+1, AMa+1, AMa+1, AMb+1, AMb+1, AMb+1, AMa+AMb+1)
        Indexed [ax, ay, az, bx, by, bz, m], relative to V_base.
        Only the m = 0 slice is physically meaningful.
    """
    max_m = AMa + AMb
    NAI_table = np.zeros((AMa + 1, AMa + 1, AMa + 1, AMb + 1, AMb + 1, AMb + 1, max_m + 1))

    def shift_NAI(shift_dict):
        """Read NAI_table at the current loop indices, shifted; 0.0 if out of bounds."""
        axis_dict = {
            "ax": 0, "ay": 1, "az": 2,
            "bx": 3, "by": 4, "bz": 5,
            "m": 6
        }

        # Captures the current loop variables from the enclosing scope.
        curr_idx = [ax, ay, az, bx, by, bz, m]
        for axis, shift in shift_dict.items():
            curr_idx[axis_dict[axis]] += shift

        # A negative angular momentum index means the term does not exist.
        if any(idx < 0 for idx in curr_idx):
            return 0.0

        return NAI_table[tuple(curr_idx)]

    # 1. Seed every Boys order for the [0,0,0|0,0,0] base case.
    U = zeta * np.sum(PC ** 2)
    for m_idx in range(max_m + 1):
        NAI_table[0, 0, 0, 0, 0, 0, m_idx] = boys_function(m_idx, U)

    # 2. Climb in total angular momentum L, so that every dependency is already filled before it is read.
    for L in range(1, AMa + AMb + 1):
        for ax in range(AMa + 1):
            for ay in range(AMa + 1 - ax):
                for az in range(AMa + 1 - ax - ay):
                    for bx in range(AMb + 1):
                        for by in range(AMb + 1 - bx):
                            for bz in range(AMb + 1 - bx - by):

                                # Only touch states on the current L shell.
                                if ax + ay + az + bx + by + bz != L:
                                    continue

                                # Orders above max_m - L are not calculated, since they can never be consumed by the recursion.
                                for m in range(max_m - L + 1):

                                    # Step up the first nonzero component found.
                                    if ax > 0:
                                        term1 = PA[0] * shift_NAI({"ax": -1})
                                        term2 = PC[0] * shift_NAI({"ax": -1, "m": +1})
                                        term3 = (0.5 / zeta) * (ax - 1) * (shift_NAI({"ax": -2}) - shift_NAI({"ax": -2, "m": +1})) if ax > 1 else 0.0
                                        term4 = (0.5 / zeta) * bx * (shift_NAI({"ax": -1, "bx": -1}) - shift_NAI({"ax": -1, "bx": -1, "m": +1})) if bx > 0 else 0.0
                                        NAI_table[ax, ay, az, bx, by, bz, m] = term1 - term2 + term3 + term4

                                    elif ay > 0:
                                        term1 = PA[1] * shift_NAI({"ay": -1})
                                        term2 = PC[1] * shift_NAI({"ay": -1, "m": +1})
                                        term3 = (0.5 / zeta) * (ay - 1) * (shift_NAI({"ay": -2}) - shift_NAI({"ay": -2, "m": +1})) if ay > 1 else 0.0
                                        term4 = (0.5 / zeta) * by * (shift_NAI({"ay": -1, "by": -1}) - shift_NAI({"ay": -1, "by": -1, "m": +1})) if by > 0 else 0.0
                                        NAI_table[ax, ay, az, bx, by, bz, m] = term1 - term2 + term3 + term4

                                    elif az > 0:
                                        term1 = PA[2] * shift_NAI({"az": -1})
                                        term2 = PC[2] * shift_NAI({"az": -1, "m": +1})
                                        term3 = (0.5 / zeta) * (az - 1) * (shift_NAI({"az": -2}) - shift_NAI({"az": -2, "m": +1})) if az > 1 else 0.0
                                        term4 = (0.5 / zeta) * bz * (shift_NAI({"az": -1, "bz": -1}) - shift_NAI({"az": -1, "bz": -1, "m": +1})) if bz > 0 else 0.0
                                        NAI_table[ax, ay, az, bx, by, bz, m] = term1 - term2 + term3 + term4

                                    # Center A indices are all 0 from here on, so the a-dependent term drops out, leaving three terms.
                                    elif bx > 0:
                                        term1 = PB[0] * shift_NAI({"bx": -1})
                                        term2 = PC[0] * shift_NAI({"bx": -1, "m": +1})
                                        term3 = (0.5 / zeta) * (bx - 1) * (shift_NAI({"bx": -2}) - shift_NAI({"bx": -2, "m": +1})) if bx > 1 else 0.0
                                        NAI_table[ax, ay, az, bx, by, bz, m] = term1 - term2 + term3

                                    elif by > 0:
                                        term1 = PB[1] * shift_NAI({"by": -1})
                                        term2 = PC[1] * shift_NAI({"by": -1, "m": +1})
                                        term3 = (0.5 / zeta) * (by - 1) * (shift_NAI({"by": -2}) - shift_NAI({"by": -2, "m": +1})) if by > 1 else 0.0
                                        NAI_table[ax, ay, az, bx, by, bz, m] = term1 - term2 + term3

                                    elif bz > 0:
                                        term1 = PB[2] * shift_NAI({"bz": -1})
                                        term2 = PC[2] * shift_NAI({"bz": -1, "m": +1})
                                        term3 = (0.5 / zeta) * (bz - 1) * (shift_NAI({"bz": -2}) - shift_NAI({"bz": -2, "m": +1})) if bz > 1 else 0.0
                                        NAI_table[ax, ay, az, bx, by, bz, m] = term1 - term2 + term3

    return NAI_table

## Cartesian component ordering

A shell of angular momentum $\ell$ contains $(\ell+1)(\ell+2)/2$ Cartesian components, and we need to enumerate them in **exactly** the order Psi4 uses to ensure the final matrices match. Psi4 orders components from the highest $x$ power downward, distributing the remainder to $y$ and then $z$. For a $d$ shell that gives $(2,0,0), (1,1,0), (1,0,1), (0,2,0), (0,1,1), (0,0,2)$, i.e. the familiar $d_{xx}, d_{xy}, d_{xz}, d_{yy}, d_{yz}, d_{zz}$.

Tutorial 11a generates the same sequence inline with a pair of nested counter loops. Here it is built into a helper so the assembly loop below stays readable.

In [6]:
def angular_combinations(n):
    """Enumerate Cartesian (nx, ny, nz) triples of total angular momentum n in Psi4's component ordering."""
    coms = []
    for nx in range(n, -1, -1):
        for ny in range(n - nx, -1, -1):
            nz = n - nx - ny
            coms.append((nx, ny, nz))
    return coms


# Validation
for am, label in [(0, "s"), (1, "p"), (2, "d")]:
    print(f"{label} shell ({len(angular_combinations(am))} components): {angular_combinations(am)}")

s shell (1 components): [(0, 0, 0)]
p shell (3 components): [(1, 0, 0), (0, 1, 0), (0, 0, 1)]
d shell (6 components): [(2, 0, 0), (1, 1, 0), (1, 0, 1), (0, 2, 0), (0, 1, 1), (0, 0, 2)]


## Setting up the molecule and basis set

By default Psi4 uses spherical harmonic ("pure angular momentum") Gaussians, whereas our recursion is written for Cartesian Gaussians, so we set `puream` to 0 for correct comparison.

In [7]:
mol = psi4.geometry("""
                        O
                        H 1 1.1
                        H 1 1.1 2 104
                        symmetry c1
                        """)

psi4.set_options({'basis':        'sto-3g',
                  'scf_type':         'pk',
                  'puream':              0,
                  'e_convergence':    1e-8,
                  'd_convergence':    1e-8})

wfn = psi4.core.Wavefunction.build(mol, psi4.core.get_global_option('basis'), quiet=True)
basis = wfn.basisset()
mints = psi4.core.MintsHelper(basis)

n_ao = basis.nao()
print(f"{basis.nshell()} shells, {n_ao} Cartesian AOs")

5 shells, 7 Cartesian AOs


## Assembling the matrices

The assembly is a three-level loop, from outside in:

1. **Shell pairs.** $\mathbf S$, $\mathbf T$, and $\mathbf V$ are all symmetric, so we only compute the $b \geq a$ blocks and fill the rest by transposition.
2. **Primitive pairs.** Everything that depends on the exponents but not on the angular momentum indices ($\zeta$, $\xi$, $\mathbf P$, $S_{\text{base}}$, $V_{\text{base}}$, and the three recursion tables) is computed once here. The recursion tables are computed once per primitive pair and then reused across every angular component pair
3. **Angular components.** The innermost loop pulls the relevant entries out of the tables and accumulates into the AO matrix.

Importantly, there is a loop over nuclear centers within the primitive pairs loop. Each nucleus contributes its own Boys-seeded table, and we accumulate them weighted by $-Z_C$ as we go. Because the nuclear charge is a scalar multiplier that does not interact with the recursion, putting it here before the angular components loop saves carrying a separate table per nucleus.

In [8]:
S = np.zeros((n_ao, n_ao))
T = np.zeros((n_ao, n_ao))
V = np.zeros((n_ao, n_ao))

# Level 1: loop over shell pairs.
for a in range(basis.nshell()):
    for b in range(basis.nshell()):

        ashell = basis.shell(a)
        bshell = basis.shell(b)

        AMa = ashell.am
        AMb = bshell.am

        # Basis function index where each shell begins.
        a_idx = ashell.function_index
        b_idx = bshell.function_index

        a_coms = angular_combinations(AMa)
        b_coms = angular_combinations(AMb)

        n_coma = len(a_coms)
        n_comb = len(b_coms)

        # S, T and V are symmetric: fill the lower block from the upper one.
        if b < a:
            S[a_idx:a_idx + n_coma, b_idx:b_idx + n_comb] = S[b_idx:b_idx + n_comb, a_idx:a_idx + n_coma].T
            T[a_idx:a_idx + n_coma, b_idx:b_idx + n_comb] = T[b_idx:b_idx + n_comb, a_idx:a_idx + n_coma].T
            V[a_idx:a_idx + n_coma, b_idx:b_idx + n_comb] = V[b_idx:b_idx + n_comb, a_idx:a_idx + n_coma].T
            continue

        # shell.ncenter gives the atom index; mol.x()/y()/z() give its coordinates.
        A = np.array([mol.x(ashell.ncenter), mol.y(ashell.ncenter), mol.z(ashell.ncenter)])
        B = np.array([mol.x(bshell.ncenter), mol.y(bshell.ncenter), mol.z(bshell.ncenter)])

        # Level 2: loop over primitive pairs.
        for a_prim in range(ashell.nprimitive):
            for b_prim in range(bshell.nprimitive):

                expa = ashell.exp(a_prim)
                expb = bshell.exp(b_prim)

                # Already carries primitive and contraction normalization.
                coefa = ashell.coef(a_prim)
                coefb = bshell.coef(b_prim)

                zeta = expa + expb
                xi = (expa * expb) / zeta
                P = (expa * A + expb * B) / zeta
                PA = P - A
                PB = P - B
                AB = A - B

                S_base = (np.pi / zeta) ** (3 / 2) * np.exp(-xi * np.sum(AB ** 2))
                V_base = 2 * (zeta / np.pi) ** 0.5 * S_base

                # Overlap calculated first, as the kinetic recursion depends on it.
                SI = get_overlap_table(PA, PB, zeta, AMa, AMb)
                TI = get_kinetic_table(PA, PB, expa, expb, AMa, AMb, SI)

                # Accumulate one Boys-seeded table per nucleus, weighted by -Z_C as we go.
                VI = np.zeros((AMa + 1, AMa + 1, AMa + 1,
                               AMb + 1, AMb + 1, AMb + 1,
                               AMa + AMb + 1))

                for C_idx in range(mol.natom()):
                    Z_C = mol.Z(C_idx)
                    C = np.array([mol.x(C_idx), mol.y(C_idx), mol.z(C_idx)])
                    PC = P - C

                    VI += get_NAI_table(PA, PB, PC, zeta, AMa, AMb) * (-Z_C)

                # Level 3: loop over angular components, reusing the tables above.
                for counta, a_com in enumerate(a_coms):
                    for countb, b_com in enumerate(b_coms):
                        x, y, z = 0, 1, 2
                        ax, ay, az = a_com
                        bx, by, bz = b_com

                        # Overlap: a clean product of the three axes.
                        S[a_idx + counta, b_idx + countb] += S_base \
                                                        * coefa * coefb \
                                                        * SI[x, ax, bx] \
                                                        * SI[y, ay, by] \
                                                        * SI[z, az, bz]

                        # Kinetic: a sum of cross terms, because the Laplacian is a sum of second derivatives.
                        t_term = (TI[x, ax, bx] * SI[y, ay, by] * SI[z, az, bz]) + \
                                 (SI[x, ax, bx] * TI[y, ay, by] * SI[z, az, bz]) + \
                                 (SI[x, ax, bx] * SI[y, ay, by] * TI[z, az, bz])

                        T[a_idx + counta, b_idx + countb] += S_base \
                                                        * coefa * coefb \
                                                        * t_term

                        # Nuclear attraction: no axis factorization at all, just a single lookup into the 7D table at m = 0.
                        V[a_idx + counta, b_idx + countb] += V_base \
                                                        * coefa * coefb \
                                                        * VI[ax, ay, az, bx, by, bz, 0]


## Validation

In [9]:
psi4.compare_values(np.asarray(mints.ao_overlap()),   S, 8, "Overlap Integrals")
psi4.compare_values(np.asarray(mints.ao_kinetic()),   T, 8, "Kinetic Energy Integrals")
psi4.compare_values(np.asarray(mints.ao_potential()), V, 8, "Nuclear Attraction Integrals")

    Overlap Integrals.....................................................................PASSED    Overlap Integrals.....................................................................PASSED

    Kinetic Energy Integrals..............................................................PASSED
    Kinetic Energy Integrals..............................................................PASSED
    Nuclear Attraction Integrals..........................................................PASSED
    Nuclear Attraction Integrals..........................................................PASSED


True

## References

1. Recursion scheme:
    > [[Obara:1986](http://aip.scitation.org/doi/abs/10.1063/1.450106)] S. Obara and A. Saika, *J. Chem. Phys.* **84**, 3963 (1986)

2. Boys function, original definition:
    > [[Boys:1950:542](http://dx.doi.org/10.1098/rspa.1950.0036)] S. F. Boys, *Proc. R. Soc. Lond. A* **200**, 542 (1950)

3. Integral algorithms and the incomplete-gamma form of the Boys function:
    > [[Helgaker:2000](https://onlinelibrary.wiley.com/doi/book/10.1002/9781119019572)] T. Helgaker, P. Jorgensen, and J. Olsen, *Molecular Electronic-Structure Theory*, John Wiley & Sons Inc, 2000